# 01 — Data Acquisition

**Purpose:** Download all raw data from the NBA API and save to `data/raw/`.

## What this notebook does
1. Sets up directory structure
2. Downloads **team game logs** for each of the 20 seasons (2005-06 → 2024-25) via `TeamGameLogs`
3. Downloads **season-level advanced team stats** via `LeagueDashTeamStats`
4. Downloads the **full league game finder** dataset via `LeagueGameFinder`
5. Saves everything as CSV to `data/raw/`
6. Confirms acquisition with shape & preview output

## ⚠️ Rate Limiting
The NBA API (stats.nba.com) enforces rate limits. This notebook uses `time.sleep(1)` between requests.  
**Do not remove the sleep calls** — you will get throttled and your IP may be temporarily blocked.

**Estimated runtime:** 10–25 minutes for a full fresh download.

## Skip condition
If `data/raw/` already contains all the expected CSV files (e.g., pulled from git), the notebook will detect them and skip re-downloading.

---
## 1. Setup — Directories & Constants

In [1]:
import os
import time
import pandas as pd
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()           # directory where this notebook lives
RAW_DIR      = NOTEBOOK_DIR / 'data' / 'raw'
GAME_LOGS_DIR  = RAW_DIR / 'game_logs'
TEAM_STATS_DIR = RAW_DIR / 'team_stats'

RAW_DIR.mkdir(parents=True, exist_ok=True)
GAME_LOGS_DIR.mkdir(parents=True, exist_ok=True)
TEAM_STATS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Raw data directory  : {RAW_DIR}')
print(f'Game logs directory : {GAME_LOGS_DIR}')
print(f'Team stats directory: {TEAM_STATS_DIR}')

# ── Season list ────────────────────────────────────────────────────────────
# NBA API season format: '2000-01', '2001-02', ..., '2024-25'
def make_season_id(start_year: int) -> str:
    return f'{start_year}-{str(start_year + 1)[-2:]}'

SEASONS = [make_season_id(y) for y in range(2000, 2025)]   # 2000-01 through 2024-25
print(f'\nSeasons to collect ({len(SEASONS)}): {SEASONS[0]} → {SEASONS[-1]}')

# ── Rate-limiting helper ────────────────────────────────────────────────────
SLEEP_BETWEEN_REQUESTS = 1.2  # seconds — be polite to stats.nba.com

Raw data directory  : C:\Users\Sejal Kotian\Documents\Penn_Fall_2025\cis5450project\cis5450-project\data\raw
Game logs directory : C:\Users\Sejal Kotian\Documents\Penn_Fall_2025\cis5450project\cis5450-project\data\raw\game_logs
Team stats directory: C:\Users\Sejal Kotian\Documents\Penn_Fall_2025\cis5450project\cis5450-project\data\raw\team_stats

Seasons to collect (25): 2000-01 → 2024-25


---
## 2. Download Team Game Logs (per season)

**Endpoint:** `TeamGameLogs`  
**What it provides:** Every team's per-game box score stats for a given season.  
One row = one team's performance in one game (both teams in a game each get a row).  
Key columns include: `TEAM_ID`, `TEAM_ABBREVIATION`, `GAME_ID`, `GAME_DATE`, `MATCHUP`, `WL`,
`PTS`, `FGM`, `FGA`, `FG_PCT`, `FG3M`, `FG3A`, `FG3_PCT`, `FTM`, `FTA`, `FT_PCT`,
`OREB`, `DREB`, `REB`, `AST`, `TOV`, `STL`, `BLK`, `PLUS_MINUS`.

In [2]:
from nba_api.stats.endpoints import TeamGameLogs

all_game_logs = []
skipped_seasons = []

for season in SEASONS:
    out_path = GAME_LOGS_DIR / f'game_logs_{season}.csv'

    # Skip if already downloaded
    if out_path.exists():
        print(f'  [SKIP] {season} — file already exists')
        skipped_seasons.append(season)
        continue

    retries = 3
    for attempt in range(retries):
        try:
            logs = TeamGameLogs(
                season_nullable=season,
                season_type_nullable='Regular Season',
            )
            df = logs.get_data_frames()[0]
            df['SEASON'] = season   # add season column for easy filtering later
            df.to_csv(out_path, index=False)
            print(f'  [OK]   {season} — {len(df):,} rows saved to {out_path.name}')
            all_game_logs.append(df)
            break
        except Exception as e:
            if attempt < retries - 1:
                print(f'  [RETRY] {season} attempt {attempt + 1} failed: {e}')
                time.sleep(5)
            else:
                print(f'  [ERR]  {season} — {e}')

    time.sleep(SLEEP_BETWEEN_REQUESTS)

# Load any skipped (already-downloaded) seasons into memory
for season in skipped_seasons:
    df = pd.read_csv(GAME_LOGS_DIR / f'game_logs_{season}.csv')
    all_game_logs.append(df)

print(f'\nLoaded game logs for {len(all_game_logs)} seasons.')

  [OK]   2000-01 — 2,378 rows saved to game_logs_2000-01.csv
  [OK]   2001-02 — 2,378 rows saved to game_logs_2001-02.csv
  [OK]   2002-03 — 2,378 rows saved to game_logs_2002-03.csv
  [OK]   2003-04 — 2,378 rows saved to game_logs_2003-04.csv
  [OK]   2004-05 — 2,460 rows saved to game_logs_2004-05.csv
  [SKIP] 2005-06 — file already exists
  [SKIP] 2006-07 — file already exists
  [SKIP] 2007-08 — file already exists
  [SKIP] 2008-09 — file already exists
  [SKIP] 2009-10 — file already exists
  [SKIP] 2010-11 — file already exists
  [SKIP] 2011-12 — file already exists
  [SKIP] 2012-13 — file already exists
  [SKIP] 2013-14 — file already exists
  [SKIP] 2014-15 — file already exists
  [SKIP] 2015-16 — file already exists
  [SKIP] 2016-17 — file already exists
  [SKIP] 2017-18 — file already exists
  [SKIP] 2018-19 — file already exists
  [SKIP] 2019-20 — file already exists
  [SKIP] 2020-21 — file already exists
  [SKIP] 2021-22 — file already exists
  [SKIP] 2022-23 — file already 

In [3]:
# Combine all seasons into a single dataframe for a quick sanity check
game_logs_all = pd.concat(all_game_logs, ignore_index=True)
print(f'Combined game logs shape: {game_logs_all.shape}')
print(f'Seasons present         : {sorted(game_logs_all["SEASON"].unique())}')
print(f'Win/Loss distribution:\n{game_logs_all["WL"].value_counts()}')
game_logs_all.head(3)

Combined game logs shape: (60048, 58)
Seasons present         : ['2000-01', '2001-02', '2002-03', '2003-04', '2004-05', '2005-06', '2006-07', '2007-08', '2008-09', '2009-10', '2010-11', '2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
Win/Loss distribution:
WL
W    30024
L    30024
Name: count, dtype: int64


,SEASON_YEAR,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,AVAILABLE_FLAG,SEASON
0,2000-01,1610612760,SEA,Seattle SuperSonics,0020001189,2001-04-18T00:00:00,SEA vs. SAS,W,48.0,43,...,4,1563,300,1909,45,160,418,3,None,2000-01
1,2000-01,1610612742,DAL,Dallas Mavericks,0020001185,2001-04-18T00:00:00,DAL vs. MIN,W,48.0,50,...,872,910,175,162,1253,1,34,136,None,2000-01
2,2000-01,1610612763,VAN,Vancouver Grizzlies,0020001188,2001-04-18T00:00:00,VAN @ GSW,W,48.0,37,...,872,652,103,2205,849,160,1142,285,None,2000-01


---
## 3. Download Season-Level Advanced Team Stats (per season)

**Endpoint:** `LeagueDashTeamStats`  
**What it provides:** Aggregated season-level stats per team including advanced metrics like
offensive rating, defensive rating, net rating, pace, true shooting %, etc.  
One row = one team's full-season summary.  
Used for season-level features: current win %, conference rank, season-normalized ratings.

In [4]:
from nba_api.stats.endpoints import LeagueDashTeamStats

all_team_stats = []
skipped_seasons_ts = []

for season in SEASONS:
    out_path = TEAM_STATS_DIR / f'team_stats_{season}.csv'

    if out_path.exists():
        print(f'  [SKIP] {season} — file already exists')
        skipped_seasons_ts.append(season)
        continue

    retries = 3
    for attempt in range(retries):
        try:
            stats = LeagueDashTeamStats(
                season=season,
                season_type_all_star='Regular Season',
                measure_type_detailed_defense='Advanced',   # Advanced = OffRtg, DefRtg, NetRtg, Pace, TS%
            )
            df = stats.get_data_frames()[0]
            df['SEASON'] = season
            df.to_csv(out_path, index=False)
            print(f'  [OK]   {season} — {len(df):,} teams saved to {out_path.name}')
            all_team_stats.append(df)
            break
        except Exception as e:
            if attempt < retries - 1:
                print(f'  [RETRY] {season} attempt {attempt + 1} failed: {e}')
                time.sleep(5)
            else:
                print(f'  [ERR]  {season} — {e}')

    time.sleep(SLEEP_BETWEEN_REQUESTS)

for season in skipped_seasons_ts:
    df = pd.read_csv(TEAM_STATS_DIR / f'team_stats_{season}.csv')
    all_team_stats.append(df)

print(f'\nLoaded advanced team stats for {len(all_team_stats)} seasons.')

  [OK]   2000-01 — 29 teams saved to team_stats_2000-01.csv
  [OK]   2001-02 — 29 teams saved to team_stats_2001-02.csv
  [OK]   2002-03 — 29 teams saved to team_stats_2002-03.csv
  [OK]   2003-04 — 29 teams saved to team_stats_2003-04.csv
  [OK]   2004-05 — 30 teams saved to team_stats_2004-05.csv
  [SKIP] 2005-06 — file already exists
  [SKIP] 2006-07 — file already exists
  [SKIP] 2007-08 — file already exists
  [SKIP] 2008-09 — file already exists
  [SKIP] 2009-10 — file already exists
  [SKIP] 2010-11 — file already exists
  [SKIP] 2011-12 — file already exists
  [SKIP] 2012-13 — file already exists
  [SKIP] 2013-14 — file already exists
  [SKIP] 2014-15 — file already exists
  [SKIP] 2015-16 — file already exists
  [SKIP] 2016-17 — file already exists
  [SKIP] 2017-18 — file already exists
  [SKIP] 2018-19 — file already exists
  [SKIP] 2019-20 — file already exists
  [SKIP] 2020-21 — file already exists
  [SKIP] 2021-22 — file already exists
  [SKIP] 2022-23 — file already exist

In [5]:
team_stats_all = pd.concat(all_team_stats, ignore_index=True)
print(f'Combined advanced team stats shape: {team_stats_all.shape}')
print(f'Columns: {list(team_stats_all.columns)}')
team_stats_all.head(3)

Combined advanced team stats shape: (746, 47)
Columns: ['TEAM_ID', 'TEAM_NAME', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'E_OFF_RATING', 'OFF_RATING', 'E_DEF_RATING', 'DEF_RATING', 'E_NET_RATING', 'NET_RATING', 'AST_PCT', 'AST_TO', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT', 'EFG_PCT', 'TS_PCT', 'E_PACE', 'PACE', 'PACE_PER40', 'POSS', 'PIE', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'OFF_RATING_RANK', 'DEF_RATING_RANK', 'NET_RATING_RANK', 'AST_PCT_RANK', 'AST_TO_RANK', 'AST_RATIO_RANK', 'OREB_PCT_RANK', 'DREB_PCT_RANK', 'REB_PCT_RANK', 'TM_TOV_PCT_RANK', 'EFG_PCT_RANK', 'TS_PCT_RANK', 'PACE_RANK', 'PIE_RANK', 'SEASON']


,TEAM_ID,TEAM_NAME,GP,W,L,W_PCT,MIN,E_OFF_RATING,OFF_RATING,E_DEF_RATING,...,AST_RATIO_RANK,OREB_PCT_RANK,DREB_PCT_RANK,REB_PCT_RANK,TM_TOV_PCT_RANK,EFG_PCT_RANK,TS_PCT_RANK,PACE_RANK,PIE_RANK,SEASON
0,1610612737,Atlanta Hawks,82,25,57,0.305,3946.0,95.6,96.9,101.9,...,29,15,12,14,28,25,26,11,26,2000-01
1,1610612738,Boston Celtics,82,36,46,0.439,3966.0,99.0,100.1,100.9,...,19,26,15,28,20,14,12,10,23,2000-01
2,1610612766,Charlotte Hornets,82,46,36,0.561,3976.0,99.3,99.7,96.3,...,7,14,1,6,10,24,23,21,7,2000-01


---
## 4. Download League Game Finder (all games, single call)

**Endpoint:** `LeagueGameFinder`  
**What it provides:** Every NBA regular-season game result across all seasons in a single request.
Useful as a flat index of all games: `GAME_ID`, `GAME_DATE`, `TEAM_ID`, `TEAM_ABBREVIATION`,
`MATCHUP`, `WL`, `PTS`, `SEASON_ID`.  
We pull all regular-season games from 2000 onwards.

In [9]:
from nba_api.stats.endpoints import LeagueGameFinder

lgf_path = RAW_DIR / 'league_game_finder.csv'

if lgf_path.exists():
    print(f'[SKIP] league_game_finder.csv already exists — loading from disk.')
    lgf_df = pd.read_csv(lgf_path)
else:
    print('Downloading league game finder data (season by season)...')
    lgf_frames = []

    for season in SEASONS:
        retries = 3
        for attempt in range(retries):
            try:
                lgf = LeagueGameFinder(
                    season_nullable=season,
                    season_type_nullable='Regular Season',
                    league_id_nullable='00',   # '00' = NBA
                )
                df = lgf.get_data_frames()[0]
                lgf_frames.append(df)
                print(f'  [OK]   {season} — {len(df):,} rows')
                break
            except Exception as e:
                if attempt < retries - 1:
                    print(f'  [RETRY] {season} attempt {attempt + 1} failed: {e}')
                    time.sleep(5)
                else:
                    print(f'  [ERR]  {season} — {e}')

        time.sleep(SLEEP_BETWEEN_REQUESTS)

    lgf_df = pd.concat(lgf_frames, ignore_index=True)

    # SEASON_ID format from this endpoint: '22000', '22001', ..., '22024' (prefix '2' = regular season)
    lgf_df['SEASON_YEAR'] = lgf_df['SEASON_ID'].astype(str).str[1:].astype(int)
    lgf_df = lgf_df[lgf_df['SEASON_YEAR'] >= 2000].copy()

    lgf_df.to_csv(lgf_path, index=False)
    print(f'[OK] Saved {len(lgf_df):,} rows to {lgf_path.name}')

print(f'League game finder shape: {lgf_df.shape}')
lgf_df.head(3)

  [OK]   2000-01 — 2,378 rows
  [OK]   2001-02 — 2,378 rows
  [OK]   2002-03 — 2,378 rows
  [OK]   2003-04 — 2,378 rows
  [OK]   2004-05 — 2,460 rows
  [OK]   2005-06 — 2,460 rows
  [OK]   2006-07 — 2,460 rows
  [OK]   2007-08 — 2,460 rows
  [OK]   2008-09 — 2,460 rows
  [OK]   2009-10 — 2,460 rows
  [OK]   2010-11 — 2,460 rows
  [OK]   2011-12 — 1,980 rows
  [OK]   2012-13 — 2,458 rows
  [OK]   2013-14 — 2,460 rows
  [OK]   2014-15 — 2,460 rows
  [OK]   2015-16 — 2,460 rows
  [OK]   2016-17 — 2,460 rows
  [OK]   2017-18 — 2,460 rows
  [OK]   2018-19 — 2,460 rows
  [OK]   2019-20 — 2,118 rows
  [OK]   2020-21 — 2,160 rows
  [OK]   2021-22 — 2,460 rows
  [OK]   2022-23 — 2,460 rows
  [OK]   2023-24 — 2,460 rows
  [OK]   2024-25 — 2,460 rows
[OK] Saved 60,048 rows to league_game_finder.csv
League game finder shape: (60048, 29)


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,SEASON_YEAR
0,22000,1610612762,UTA,Utah Jazz,0020001187,2001-04-18,UTA @ PHX,L,240,93,...,13,22,35,25,8,2,20,27,-5.0,2000
1,22000,1610612760,SEA,Seattle SuperSonics,0020001189,2001-04-18,SEA vs. SAS,W,240,105,...,9,41,50,26,6,8,5,14,38.0,2000
2,22000,1610612742,DAL,Dallas Mavericks,0020001185,2001-04-18,DAL vs. MIN,W,241,120,...,6,39,45,33,8,9,14,23,20.0,2000


---
## 5. Acquisition Summary

Confirm all expected files are present in `data/raw/`.

In [10]:
print('=== Files in data/raw/ ===')
print(f'\n  league_game_finder.csv')

print(f'\n  game_logs/ ({len(list(GAME_LOGS_DIR.glob("*.csv")))} files)')
for f in sorted(GAME_LOGS_DIR.glob('*.csv')):
    print(f'    {f.name}  ({f.stat().st_size / 1_000_000:.2f} MB)')

print(f'\n  team_stats/ ({len(list(TEAM_STATS_DIR.glob("*.csv")))} files)')
for f in sorted(TEAM_STATS_DIR.glob('*.csv')):
    print(f'    {f.name}  ({f.stat().st_size / 1_000_000:.2f} MB)')

# ── Completeness checks ────────────────────────────────────────────────────
expected_game_log_files = {f'game_logs_{s}.csv' for s in SEASONS}
actual_game_log_files   = {f.name for f in GAME_LOGS_DIR.glob('*.csv')}
missing = expected_game_log_files - actual_game_log_files
if missing:
    print(f'\n⚠️  Missing game log files: {sorted(missing)}')
else:
    print(f'\n✅  All {len(SEASONS)} season game log files present.')

expected_ts_files = {f'team_stats_{s}.csv' for s in SEASONS}
actual_ts_files   = {f.name for f in TEAM_STATS_DIR.glob('*.csv')}
missing_ts = expected_ts_files - actual_ts_files
if missing_ts:
    print(f'⚠️  Missing team stats files: {sorted(missing_ts)}')
else:
    print(f'✅  All {len(SEASONS)} season team stats files present.')

if (RAW_DIR / 'league_game_finder.csv').exists():
    print('✅  league_game_finder.csv present.')
else:
    print('⚠️  league_game_finder.csv is MISSING.')

=== Files in data/raw/ ===

  league_game_finder.csv

  game_logs/ (25 files)
    game_logs_2000-01.csv  (0.66 MB)
    game_logs_2001-02.csv  (0.66 MB)
    game_logs_2002-03.csv  (0.66 MB)
    game_logs_2003-04.csv  (0.66 MB)
    game_logs_2004-05.csv  (0.69 MB)
    game_logs_2005-06.csv  (0.69 MB)
    game_logs_2006-07.csv  (0.69 MB)
    game_logs_2007-08.csv  (0.69 MB)
    game_logs_2008-09.csv  (0.69 MB)
    game_logs_2009-10.csv  (0.69 MB)
    game_logs_2010-11.csv  (0.69 MB)
    game_logs_2011-12.csv  (0.55 MB)
    game_logs_2012-13.csv  (0.70 MB)
    game_logs_2013-14.csv  (0.70 MB)
    game_logs_2014-15.csv  (0.70 MB)
    game_logs_2015-16.csv  (0.70 MB)
    game_logs_2016-17.csv  (0.70 MB)
    game_logs_2017-18.csv  (0.70 MB)
    game_logs_2018-19.csv  (0.70 MB)
    game_logs_2019-20.csv  (0.60 MB)
    game_logs_2020-21.csv  (0.61 MB)
    game_logs_2021-22.csv  (0.70 MB)
    game_logs_2022-23.csv  (0.70 MB)
    game_logs_2023-24.csv  (0.70 MB)
    game_logs_2024-25.csv  (0.70 M

In [11]:
# Final row count summary
print('=== Row count summary ===')
print(f'  game_logs combined   : {len(game_logs_all):>8,} rows  (target: ~65,000+)')
print(f'  team_stats combined  : {len(team_stats_all):>8,} rows  (30 teams × 25 seasons = ~750)')
print(f'  league_game_finder   : {len(lgf_df):>8,} rows')

print('\n=== game_logs column list ===')
print(list(game_logs_all.columns))

print('\n=== Seasons in game_logs ===')
print(sorted(game_logs_all['SEASON'].unique()))

=== Row count summary ===
  game_logs combined   :   60,048 rows  (target: ~65,000+)
  team_stats combined  :      746 rows  (30 teams × 25 seasons = ~750)
  league_game_finder   :   60,048 rows

=== game_logs column list ===
['SEASON_YEAR', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK', 'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK', 'AVAILABLE_FLAG', 'SEASON']

=== Seasons in game_logs ===
['2000-01', '2001-02', '2002-03', '2003-04', '2004-05', '2005-06', '2006-07', '2007-08', '2008-09', '2009-10'

---
## 6. Next Steps

Data acquisition is complete. Proceed to:

- **`02_eda.ipynb`**  — Exploratory data analysis, distributions, and hypothesis testing:
  - Test 1: Home court advantage (one-sample proportion z-test + permutation test)
  - Test 2: Rest day effect (permutation test on win rate difference)

- **`03_cleaning_features.ipynb`**  — Cleaning, wrangling, and feature engineering:
  - Rolling 5-game and 10-game averages per team
  - Advanced metric joins
  - Back-to-back flags, rest days, win streak features
  - Opponent strength features
  - Season-normalized features to account for era drift

- **`04_modeling.ipynb`**  — Modeling and evaluation:
  - Logistic Regression, Random Forest, XGBoost, Ensemble
  - Time-aware cross-validation (train on earlier seasons, test on later)
  - Metrics: accuracy, AUC-ROC, precision, recall, F1
  - Baseline comparison: home-team-always-wins (~60%)